In [1]:
!apt-get update
!apt-get install -y openmpi-bin libopenmpi-dev
!mpirun --version

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,665 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,717 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jamm

In [3]:
!apt-get update
!apt-get install -y g++ libomp-dev

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading

# OPENMP

## EXO 1 OPENMP


In [4]:
%%writefile openmp_schedule.c
#include <stdio.h>
#include <omp.h>

int main() {
    int num_threads, num_iterations;
    printf("Entrez le nombre de threads : ");
    scanf("%d", &num_threads);
    printf("Entrez le nombre d'itérations : ");
    scanf("%d", &num_iterations);
    omp_set_num_threads(num_threads);

    int sum_static = 0, sum_dynamic = 0, sum_guided = 0;
    double start_time, end_time;
    double time_static, time_dynamic, time_guided;

    printf("\n--- Ordonnancement STATIC ---\n");
    start_time = omp_get_wtime();
    #pragma omp parallel
    {
        #pragma omp for schedule(static, 4)
        for (int i = 0; i < num_iterations; i++) {
            printf("Itération %d, Thread %d/%d\n", i, omp_get_thread_num(), omp_get_num_threads());
            #pragma omp atomic
            sum_static += i;
        }
    }
    end_time = omp_get_wtime();
    time_static = end_time - start_time;

    printf("\n--- Ordonnancement DYNAMIC ---\n");
    start_time = omp_get_wtime();
    #pragma omp parallel
    {
        #pragma omp for schedule(dynamic, 4)
        for (int i = 0; i < num_iterations; i++) {
            printf("Itération %d, Thread %d/%d\n", i, omp_get_thread_num(), omp_get_num_threads());
            #pragma omp atomic
            sum_dynamic += i;
        }
    }
    end_time = omp_get_wtime();
    time_dynamic = end_time - start_time;

    printf("\n--- Ordonnancement GUIDED ---\n");
    start_time = omp_get_wtime();
    #pragma omp parallel
    {
        #pragma omp for schedule(guided, 4)
        for (int i = 0; i < num_iterations; i++) {
            printf("Itération %d, Thread %d/%d\n", i, omp_get_thread_num(), omp_get_num_threads());
            #pragma omp atomic
            sum_guided += i;
        }
    }
    end_time = omp_get_wtime();
    time_guided = end_time - start_time;

    printf("\n--- Résultats ---\n");
    printf("Somme (static) : %d\n", sum_static);
    printf("Somme (dynamic) : %d\n", sum_dynamic);
    printf("Somme (guided) : %d\n", sum_guided);
    printf("Temps static : %.6f secondes\n", time_static);
    printf("Temps dynamic : %.6f secondes\n", time_dynamic);
    printf("Temps guided : %.6f secondes\n", time_guided);

    return 0;
}

Writing openmp_schedule.c


In [5]:
%%writefile input.txt
4
12

Writing input.txt


In [6]:
!g++ -fopenmp openmp_schedule.c -o openmp_schedule

In [7]:
!./openmp_schedule < input.txt

Entrez le nombre de threads : Entrez le nombre d'itérations : 
--- Ordonnancement STATIC ---
Itération 0, Thread 0/4
Itération 1, Thread 0/4
Itération 2, Thread 0/4
Itération 3, Thread 0/4
Itération 8, Thread 2/4
Itération 9, Thread 2/4
Itération 10, Thread 2/4
Itération 11, Thread 2/4
Itération 4, Thread 1/4
Itération 5, Thread 1/4
Itération 6, Thread 1/4
Itération 7, Thread 1/4

--- Ordonnancement DYNAMIC ---
Itération 0, Thread 0/4
Itération 1, Thread 0/4
Itération 2, Thread 0/4
Itération 3, Thread 0/4
Itération 8, Thread 0/4
Itération 9, Thread 0/4
Itération 10, Thread 0/4
Itération 11, Thread 0/4
Itération 4, Thread 2/4
Itération 5, Thread 2/4
Itération 6, Thread 2/4
Itération 7, Thread 2/4

--- Ordonnancement GUIDED ---
Itération 0, Thread 0/4
Itération 1, Thread 0/4
Itération 2, Thread 0/4
Itération 3, Thread 0/4
Itération 8, Thread 0/4
Itération 9, Thread 0/4
Itération 10, Thread 0/4
Itération 11, Thread 0/4
Itération 4, Thread 2/4
Itération 5, Thread 2/4
Itération 6, Thread 2/

## EXO 2 OPENMP

In [10]:
%%writefile array_sum.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <omp.h>

#define ARRAY_SIZE 100
#define NUM_THREADS 4

int main() {
    srand(time(NULL));
    int array[ARRAY_SIZE];
    int sum = 0;

    printf("====================================\n");
    printf("Programme OpenMP - Exercice 2\n");
    printf("====================================\n");
    printf("Taille du tableau : %d\n", ARRAY_SIZE);
    printf("Nombre de threads : %d\n", NUM_THREADS);
    printf("====================================\n");

    omp_set_num_threads(NUM_THREADS);

    #pragma omp parallel
    {
        #pragma omp single
        {
            printf("Initialisation du tableau par un seul thread (Thread %d)\n", omp_get_thread_num());
            for (int i = 0; i < ARRAY_SIZE; i++) {
                array[i] = rand() % 100;
            }
            printf("Tableau initialisé avec %d valeurs aléatoires.\n", ARRAY_SIZE);
        }

        int local_sum = 0;
        #pragma omp for
        for (int i = 0; i < ARRAY_SIZE; i++) {
            local_sum += array[i];
        }

        #pragma omp critical
        {
            printf("Thread %d a calculé une somme locale = %d\n", omp_get_thread_num(), local_sum);
        }

        #pragma omp atomic
        sum += local_sum;

        #pragma omp barrier

        #pragma omp master
        {
            printf("====================================\n");
            printf("Résultat final\n");
            printf("====================================\n");
            printf("Somme totale des éléments : %d\n", sum);
            printf("====================================\n");
        }
    }

    return 0;
}

Overwriting array_sum.c


In [11]:
!g++ -fopenmp array_sum.c -o array_sum

In [12]:
!./array_sum

Programme OpenMP - Exercice 2
Taille du tableau : 100
Nombre de threads : 4
Initialisation du tableau par un seul thread (Thread 2)
Tableau initialisé avec 100 valeurs aléatoires.
Thread 3 a calculé une somme locale = 1177
Thread 2 a calculé une somme locale = 1423
Thread 1 a calculé une somme locale = 1144
Thread 0 a calculé une somme locale = 1402
Résultat final
Somme totale des éléments : 5146


## EXO 3 OPENMP

In [17]:
%%writefile matrix_sum.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <omp.h>

#define NUM_THREADS 4
#define N 10

int main() {
    srand(time(NULL));
    int **matrix = (int**)malloc(N * sizeof(int *));
    for (int i = 0; i < N; i++) {
        matrix[i] = (int*)malloc(N * sizeof(int));
    }
    int sum = 0;

    printf("====================================\n");
    printf("Programme OpenMP - Exercice 3\n");
    printf("====================================\n");
    printf("Nombre de threads : %d\n", NUM_THREADS);
    printf("Taille de la matrice : %d x %d\n", N, N);
    printf("====================================\n");

    omp_set_nested(1);
    omp_set_num_threads(NUM_THREADS);

    #pragma omp parallel
    {
        #pragma omp single
        {
            printf("Initialisation de la matrice par le thread %d\n", omp_get_thread_num());
            for (int i = 0; i < N; i++) {
                for (int j = 0; j < N; j++) {
                    matrix[i][j] = rand() % 10;
                }
            }
            printf("Matrice initialisée avec %d x %d valeurs aléatoires.\n", N, N);
        }
    }

    printf("\nMatrice initiale :\n");
    printf("-----------------\n");
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            printf("%2d ", matrix[i][j]);
        }
        printf("\n");
    }
    printf("-----------------\n");

    #pragma omp parallel
    {
        int local_sum = 0;
        #pragma omp for
        for (int i = 0; i < N; i++) {
            #pragma omp parallel for
            for (int j = 0; j < N; j++) {
                local_sum += matrix[i][j];
            }
        }
        #pragma omp atomic
        sum += local_sum;
        #pragma omp critical
        {
            printf("Thread %d a calculé une somme locale = %d\n", omp_get_thread_num(), local_sum);
        }
    }

    printf("\n====================================\n");
    printf("Résultat final\n");
    printf("====================================\n");
    printf("Somme totale des éléments : %d\n", sum);
    printf("====================================\n");

    for (int i = 0; i < N; i++) {
        free(matrix[i]);
    }
    free(matrix);

    return 0;
}

Overwriting matrix_sum.c


In [18]:
!g++ -fopenmp matrix_sum.c -o matrix_sum

In [19]:
!./matrix_sum

Programme OpenMP - Exercice 3
Nombre de threads : 4
Taille de la matrice : 10 x 10
Initialisation de la matrice par le thread 0
Matrice initialisée avec 10 x 10 valeurs aléatoires.

Matrice initiale :
-----------------
 2  0  9  0  3  9  9  7  3  7 
 3  8  9  7  6  6  1  8  4  7 
 4  2  9  1  4  9  8  3  3  6 
 7  6  9  6  6  2  5  8  1  8 
 7  5  9  6  2  5  4  5  4  0 
 4  0  3  3  1  7  4  9  3  0 
 8  0  6  7  8  4  1  3  4  4 
 3  1  1  2  9  5  0  5  0  6 
 6  7  6  9  0  9  8  7  9  3 
 7  9  5  5  8  3  1  9  6  6 
-----------------
Thread 1 a calculé une somme locale = 139
Thread 3 a calculé une somme locale = 123
Thread 2 a calculé une somme locale = 77
Thread 0 a calculé une somme locale = 157

Résultat final
Somme totale des éléments : 496


## EXO 4 OPENMP

In [31]:
%%writefile matrix_multiply.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>
#include <omp.h>

void sequential_multiply(double **A, double **B, double **C, int N) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            C[i][j] = 0;
            for (int k = 0; k < N; k++) {
                C[i][j] += A[i][k] * B[k][j];
            }
        }
    }
}

void block_multiply(double **A, double **B, double **C, int N, int block_size) {
    int block_count = 0;
    #pragma omp parallel
    {
        #pragma omp for collapse(2)
        for (int ii = 0; ii < N; ii += block_size) {
            for (int jj = 0; jj < N; jj += block_size) {
                #pragma omp task
                {
                    for (int i = ii; i < ii + block_size && i < N; i++) {
                        for (int j = jj; j < jj + block_size && j < N; j++) {
                            for (int k = 0; k < N; k++) {
                                C[i][j] += A[i][k] * B[k][j];
                            }
                        }
                    }
                    #pragma omp atomic
                    block_count++;
                }
            }
        }
    }
    printf("%d\n", block_count); // Display total blocks processed
}

void print_matrix_with_blocks(double **M, int N, int block_size, const char *name) {
    printf("\nMatrice %s (avec blocs de taille %dx%d):\n", name, block_size, block_size);
    printf("-----------------\n");
    for (int ii = 0; ii < N; ii += block_size) {
        for (int i = ii; i < ii + block_size && i < N; i++) {
            for (int jj = 0; jj < N; jj += block_size) {
                for (int j = jj; j < jj + block_size && j < N; j++) {
                    printf("%6.2f ", M[i][j]);
                }
                if (jj + block_size < N) printf("| "); // Block separator
            }
            printf("\n");
        }
        if (ii + block_size < N) {
            for (int j = 0; j < N + (N/block_size - 1); j++) printf("---");
            printf("\n");
        }
    }
    printf("-----------------\n");
}

double sum_matrix(double **C, int N) {
    double sum = 0;
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            sum += C[i][j];
        }
    }
    return sum;
}

int main() {
    int N, block_size;
    printf("Entrez la taille de la matrice (N) : ");
    scanf("%d", &N);
    printf("Entrez la taille du bloc (block_size) : ");
    scanf("%d", &block_size);

    // Allocate matrices
    double **A = (double**)malloc(N * sizeof(double*));
    double **B = (double**)malloc(N * sizeof(double*));
    double **C_seq = (double**)malloc(N * sizeof(double*));
    double **C_par = (double**)malloc(N * sizeof(double*));
    for (int i = 0; i < N; i++) {
        A[i] = (double*)malloc(N * sizeof(double));
        B[i] = (double*)malloc(N * sizeof(double));
        C_seq[i] = (double*)malloc(N * sizeof(double));
        C_par[i] = (double*)malloc(N * sizeof(double));
    }

    double time_seq, time_par, start, end;
    double sum_seq, sum_par;

    #pragma omp parallel
    {
        #pragma omp sections
        {
            #pragma omp section
            {
                printf("Initialisation des matrices par le thread %d\n", omp_get_thread_num());
                srand(time(NULL));
                for (int i = 0; i < N; i++) {
                    for (int j = 0; j < N; j++) {
                        A[i][j] = (double)(rand() % 100) / 10.0;
                        B[i][j] = (double)(rand() % 100) / 10.0;
                        C_seq[i][j] = 0;
                        C_par[i][j] = 0;
                    }
                }
                printf("Matrices A et B initialisées avec %d x %d valeurs.\n", N, N);
            }
            #pragma omp section
            {
                printf("Calcul séquentiel par le thread %d\n", omp_get_thread_num());
                start = omp_get_wtime();
                sequential_multiply(A, B, C_seq, N);
                end = omp_get_wtime();
                time_seq = end - start;
            }
            #pragma omp section
            {
                printf("Calcul parallèle par blocs par le thread %d\n", omp_get_thread_num());
                start = omp_get_wtime();
                block_multiply(A, B, C_par, N, block_size);
                end = omp_get_wtime();
                time_par = end - start;
            }
        }
    }

    // Display matrices with block structure (only for small N)
    if (N <= 8) {
        print_matrix_with_blocks(A, N, block_size, "A");
        print_matrix_with_blocks(B, N, block_size, "B");
        print_matrix_with_blocks(C_seq, N, block_size, "C (séquentiel)");
        print_matrix_with_blocks(C_par, N, block_size, "C (parallèle)");
    }

    sum_seq = sum_matrix(C_seq, N);
    sum_par = sum_matrix(C_par, N);

    printf("\n====================================\n");
    printf("Résultats\n");
    printf("====================================\n");
    printf("Temps séquentiel : %.6f secondes\n", time_seq);
    printf("Temps parallèle : %.6f secondes\n", time_par);
    printf("Somme des éléments (séquentiel) : %.2f\n", sum_seq);
    printf("Somme des éléments (parallèle) : %.2f\n", sum_par);
    printf("====================================\n");

    // Free matrices
    for (int i = 0; i < N; i++) {
        free(A[i]); free(B[i]); free(C_seq[i]); free(C_par[i]);
    }
    free(A); free(B); free(C_seq); free(C_par);

    return 0;
}

Overwriting matrix_multiply.c


In [33]:
!g++ -fopenmp matrix_multiply.c -o matrix_multiply

In [32]:
%%writefile input.txt
4
2

Overwriting input.txt


In [34]:
!./matrix_multiply < input.txt

Entrez la taille de la matrice (N) : Entrez la taille du bloc (block_size) : Initialisation des matrices par le thread 0
Matrices A et B initialisées avec 4 x 4 valeurs.
Calcul parallèle par blocs par le thread 0
Calcul séquentiel par le thread 1
4

Matrice A (avec blocs de taille 2x2):
-----------------
  4.30   0.40 |   9.20   1.00 
  1.00   5.80 |   4.70   7.00 
---------------
  9.60   2.70 |   5.40   2.50 
  5.00   2.50 |   0.30   0.70 
-----------------

Matrice B (avec blocs de taille 2x2):
-----------------
  1.90   1.40 |   1.00   2.00 
  7.50   7.60 |   2.20   5.60 
---------------
  6.30   6.30 |   8.00   9.30 
  6.10   7.50 |   0.50   9.90 
-----------------

Matrice C (séquentiel) (avec blocs de taille 2x2):
-----------------
 75.23  74.52 |  79.28 106.30 
117.71 127.59 |  54.86 147.49 
---------------
 87.76  86.73 |  59.99 109.29 
 34.41  33.14 |  13.25  33.72 
-----------------

Matrice C (parallèle) (avec blocs de taille 2x2):
-----------------
 75.23  74.52 |  79.28 1

# MPI

## EXO1

In [51]:
%%writefile premier_mpi.c
#include <stdio.h>
#include <mpi.h>

int main(int argc, char** argv) {
    int rang, taille;

    MPI_Init(&argc, &argv);
    MPI_Comm_rank(MPI_COMM_WORLD, &rang);
    MPI_Comm_size(MPI_COMM_WORLD, &taille);
    printf("Je suis le processus %d parmi %d processus\n", rang, taille);
    MPI_Finalize();

    return 0;
}

Overwriting premier_mpi.c


In [52]:
!mpicc premier_mpi.c -o premier_mpi

In [53]:
!mpirun --allow-run-as-root --oversubscribe -np 4 ./premier_mpi

Je suis le processus 1 parmi 4 processus
Je suis le processus 3 parmi 4 processus
Je suis le processus 2 parmi 4 processus
Je suis le processus 0 parmi 4 processus


## EXO 2